# Day 2 - Python Exercises: Passwords and Hashing
# Cyber Squad - U.S. Cyber Range edition
# Run each cell with Shift + Enter. Fill in the blanks marked ___.

In [ ]:
# Exercise 1: Keyspace
N, L = 26, 8
keyspace = ___  # formula: N to the power L
print("keyspace:", keyspace)
rate = 1_000_000_000
print("seconds to crack:", keyspace / ___)  # divide keyspace by rate

In [ ]:
# Exercise 2: Hashing
import hashlib
h1 = hashlib.sha256(b"sunshine").hexdigest()
h2 = hashlib.sha256(b"Sunshine").hexdigest()
print("hash of sunshine:", h1)
print("hash of Sunshine:", h2)
print("same?", h1 == ___)  # compare h1 and h2

In [ ]:
# Challenge: add a salt
import secrets
salt = secrets.token_hex(___)  # generate 8 random bytes of salt (pass integer 8)
pw = "sunshine"
salted = (salt + pw).encode()
print("salted hash:", hashlib.sha256(___).hexdigest())  # hash the salted bytes

## Going further: randomness and password cracking

Building on the exercises above, these cover secure randomness, generating strong passwords, and how weak ones get cracked.

### Python random Module: Basic Operations


In [ ]:
import random

# Basic operations
print('randint(1, 100):', random.randint(1, 100))
print('choice([a,b,c]):', random.choice(['alpha', 'beta', 'gamma']))
print('uniform(0, 1)  :', random.uniform(0, 1))

data = list(range(10))
random.shuffle(data)
print('After shuffle  :', data)


### Seed Reproducibility Demo

When you set a seed, the sequence is **completely reproducible**.
This is useful for testing but **catastrophic** if used for keys or tokens.


In [ ]:
import random

print('Run 1 with seed 42:')
random.seed(42)
seq1 = [random.randint(0, 100) for _ in range(5)]
print(' ', seq1)

print('Run 2 with seed 42 (identical!):')
random.seed(42)
seq2 = [random.randint(0, 100) for _ in range(5)]
print(' ', seq2)

print(f'Sequences identical: {seq1 == seq2}')  # Expected: True

print('\nNo seed (different each run):')
random.seed(None)
print(' ', [random.randint(0, 100) for _ in range(5)])


### /dev/random vs /dev/urandom

On Linux/macOS, the OS provides two random devices:

- `/dev/random`: blocks until enough entropy is gathered from hardware events.
  Very slow. Historically more conservative but now equivalent on modern kernels.
- `/dev/urandom`: non-blocking, uses a CSPRNG seeded from OS entropy.
  Fast and cryptographically secure for all practical purposes.

Python's `os.urandom()` calls `/dev/urandom` (or the Windows equivalent).


In [ ]:
import os

# os.urandom -- cryptographically secure, non-blocking
random_bytes = os.urandom(16)
print(f'os.urandom(16) hex: {random_bytes.hex()}')
print(f'os.urandom(16) int: {int.from_bytes(random_bytes, "big")}')

# Reading /dev/urandom directly (Unix only)
import platform
if platform.system() != 'Windows':
    with open('/dev/urandom', 'rb') as f:
        dev_bytes = f.read(8)
    print(f'/dev/urandom 8 bytes: {dev_bytes.hex()}')
else:
    print('Windows: /dev/urandom not available, os.urandom() is the equivalent')


### SystemRandom: OS-Level Randomness via random API


In [ ]:
import random, time

sr = random.SystemRandom()   # Uses os.urandom() under the hood
print('SystemRandom values (non-reproducible):')
print([sr.randint(0, 999) for _ in range(5)])

# Demonstrate: seeding SystemRandom has NO EFFECT
sr.seed(42)   # silently ignored
run1 = [sr.randint(0, 100) for _ in range(5)]
sr.seed(42)
run2 = [sr.randint(0, 100) for _ in range(5)]
print(f'\nSeed ignored -- runs differ: {run1 != run2}')
print(f'Run 1: {run1}')
print(f'Run 2: {run2}')


### Student Challenge: Secure Password Generator

Use the `secrets` module (not `random`) to build a password generator that:
- Produces passwords of at least 16 characters
- Guarantees at least one lowercase, uppercase, digit, and special character
- Is cryptographically secure


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Implement a secure random password generator
#
# Requirements:
#   - Use secrets module (NOT random)
#   - Length >= 16 (default)
#   - Must include: lowercase, uppercase, digit, special char
#   - Should reject passwords that don't satisfy all requirements
#     (keep regenerating until all checks pass)
#
# Expected output example: 'G@7kMpX!2nRvLs#8'
# ==========================================

import secrets, string

def generate_secure_password(length: int = 16) -> str:
    """
    Generate a cryptographically secure random password.
    Guarantees at least one char from each required character class.
    """
    # TODO: implement this function
    # Hint:
    #   alphabet = string.ascii_lowercase + string.ascii_uppercase + ...
    #   Use a while loop to keep generating until all requirements met
    #   Use secrets.choice(alphabet) for each character
    pass

# Test it:
for i in range(5):
    pwd = generate_secure_password()
    if pwd:  # remove 'if pwd' once implemented
        print(f'Password {i+1}: {pwd} (length: {len(pwd)})')


### Dictionary Attack on MD5 Hashes

Dictionary attacks try common passwords from a wordlist.
This is why storing unsalted MD5 hashes is dangerous.


In [ ]:
import hashlib

# Simulated stolen hash database (unsalted MD5)
stolen_hashes = {
    'alice':   '5f4dcc3b5aa765d61d8327deb882cf99',  # password
    'bob':     '827ccb0eea8a706c4c34a16891f84e7b',  # 12345678
    'charlie': 'e99a18c428cb38d5f260853678922e03',  # abc123
    'dave':    'b14a7b8059d9c055954c92674ce60032',  # won't crack
}

# Common passwords wordlist
wordlist = [
    'password', 'password123', '123456', '12345678', 'qwerty',
    'abc123', 'letmein', 'monkey', 'dragon', 'master',
    'iloveyou', 'sunshine', 'princess', 'football', 'shadow'
]

def dictionary_attack_md5(target_hash: str, wordlist: list) -> str:
    """Try each word in wordlist. Return the word if found, else None."""
    for word in wordlist:
        if hashlib.md5(word.encode()).hexdigest() == target_hash:
            return word
    return None

print('Dictionary attack results:')
print(f'{"Username":<12} {"Hash (first 16...)":<20} {"Cracked Password"}')
print('-' * 55)
for user, h in stolen_hashes.items():
    cracked = dictionary_attack_md5(h, wordlist)
    status  = cracked if cracked else '(not in wordlist)'
    print(f'{user:<12} {h[:16]}...  {status}')
